In [ ]:
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from tensorflow.keras.callbacks import ReduceLROnPlateau
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

In [ ]:
df_train = pd.read_csv('datasets/train.csv')
df_valid = pd.read_csv('datasets/valid.csv')
df_test = pd.read_csv('datasets/test.csv')

In [ ]:
df_train.head()

In [ ]:
df_train.head()

In [ ]:
for i in df_train.columns:
    if df_train[i].dtype == 'object':
        df_train[i] = df_train[i].fillna('Unknown')
    else:
        df_train[i] = df_train[i].fillna(df_train[i].mean())
for i in df_valid.columns:
    if df_valid[i].dtype == 'object':
        df_valid[i] = df_valid[i].fillna('Unknown')
    else:
        df_valid[i] = df_valid[i].fillna(df_valid[i].mean())
for i in df_test.columns:
    if df_test[i].dtype == 'object':
        df_test[i] = df_test[i].fillna('Unknown')
    else:
        df_test[i] = df_test[i].fillna(df_test[i].mean())
        
        

In [ ]:
mapping1 = {
    "low": 0,
    "medium": 1,
    "high": 2
}
df_train['soil_liquefaction_risk'] = df_train['soil_liquefaction_risk'] .map(mapping1)
df_valid['soil_liquefaction_risk'] = df_valid['soil_liquefaction_risk'] .map(mapping1)
df_test['soil_liquefaction_risk'] = df_test['soil_liquefaction_risk'] .map(mapping1)

In [ ]:
df_train.head()

In [ ]:
mapping2 = {
    "no": 0,
    "yes": 1
}
df_train['Shear_wall'] = df_train['Shear_wall'].map(mapping2)
df_valid['Shear_wall'] = df_valid['Shear_wall'].map(mapping2)
df_test['Shear_wall'] = df_test['Shear_wall'].map(mapping2)

In [ ]:
df_train.head()

In [ ]:
df_train = pd.get_dummies(df_train, columns = ['structure_type', 'soil_type'])
df_valid = pd.get_dummies(df_valid, columns = ['structure_type', 'soil_type'])
df_test = pd.get_dummies(df_test, columns = ['structure_type', 'soil_type'])

In [ ]:
df_valid = df_valid.reindex(columns = df_train.columns, fill_value = 0)
df_test = df_test.reindex(columns = df_train.columns, fill_value = 0)

In [ ]:
df_train.head(50)

In [ ]:
def add_features(df):
    df['seismic_stress'] = df['earthquake_magnitude'] * df['floors']
    df['building_mass_proxy'] = df['area'] * df['floors']
    df['age_height_ratio'] = df['building_age'] / (df['floors'] + 1)

    df['structural_risk_index'] = (
        df['earthquake_magnitude'] *
        df['floors'] *
        (df['building_age'] + 1)
    )
    df['seismic_energy'] = df['earthquake_magnitude'] ** 2 * df['floors']
    df['liquefaction_seismic'] = df['soil_liquefaction_risk'] * df['earthquake_magnitude']
    df['shear_wall_risk'] = (1- df['Shear_wall']) * df['earthquake_magnitude']
    df['floors_area_density'] = df['floors'] / (df['area'] + 1)
    return df

In [ ]:
df_train = add_features(df_train)
df_valid = add_features(df_valid)
df_test = add_features(df_test)

In [ ]:
X_train = df_train.drop('collapse_probablity', axis = 1)
y_train = df_train['collapse_probablity']

X_valid = df_valid.drop('collapse_probablity', axis = 1)
y_valid = df_valid['collapse_probablity']

X_test = df_test.drop('collapse_probablity', axis = 1)
y_test = df_test['collapse_probablity']

y_train = y_train / 100.0
y_valid = y_valid / 100.0
y_test = y_test / 100.0

In [ ]:
joblib.dump(X_train.columns.tolist(), 'model_columns.pkl')
print("columns saved:", X_train.columns.tolist())

In [ ]:
scaler = StandardScaler()

num_cols = ['floors', 'building_age', 'area', 'earthquake_magnitude', 'seismic_stress', 'building_mass_proxy', 'age_height_ratio', 'structural_risk_index', 'seismic_energy','liquefaction_seismic', 'shear_wall_risk', 'floors_area_density']

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_valid[num_cols] = scaler.transform(X_valid[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [ ]:
model = keras.Sequential([
    layers.Input(shape = (23,)),

    layers.Dense(512, activation = 'swish'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(384, activation = 'swish'),
    layers.BatchNormalization(),
    layers.Dropout(0.25),

    layers.Dense(256, activation = 'swish'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(192, activation = 'swish'),
    layers.BatchNormalization(),
    layers.Dropout(0.15),

    layers.Dense(128, activation = 'swish'),
    layers.BatchNormalization(),
    layers.Dropout(0.1),

    layers.Dense(64, activation = 'swish'),
    layers.BatchNormalization(),
    layers.Dropout(0.05),

    layers.Dense(32, activation = 'swish'),

    layers.Dense(1, activation = 'sigmoid')
])
model.summary()

In [ ]:
model.compile(
    optimizer = keras.optimizers.Adam(learning_rate = 1e-3, clipnorm = 1.0),
    loss = keras.losses.Huber(delta = 5.0),
    metrics = ['mae']
)

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor = 'val_loss',
        patience = 70,
        restore_best_weights = True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor = 'val_loss',
        factor = 0.7,
        patience = 20,
        min_lr = 1e-7,
        verbose = 1
    ),
    keras.callbacks.ModelCheckpoint(
        'DL_model.keras',
        monitor = 'val_loss',
        save_best_only = True,
        verbose = 1
    )
     
]

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data = (X_valid, y_valid),
    epochs = 1000,
    batch_size = 1024,
    callbacks = callbacks
)

In [ ]:
joblib.dump(scaler, 'scaler.pkl')

In [ ]:
print("model info")
print(f"Total epochs trained: {len(history.history['loss'])}")
print(f"Best val_loss: {min(history.history['val_loss']):.4f} at epoch {history.history['val_loss'].index(min(history.history['val_loss'])) + 1}")
print(f"Best val_mae: {min(history.history['val_mae']):.4f} at epoch {history.history['val_mae'].index(min(history.history['val_mae'])) + 1}")
print(f"final learning rate: {history.history['learning_rate'][-1]:.6f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

# MAE
axes[1].plot(history.history['mae'], label='Train MAE')
axes[1].plot(history.history['val_mae'], label='Val MAE')
axes[1].set_title('MAE')
axes[1].set_xlabel('Epoch')
axes[1].legend()

# Learning Rate
axes[2].plot(history.history['learning_rate'])
axes[2].set_title('Learning Rate')
axes[2].set_xlabel('Epoch')
axes[2].set_yscale('log')

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()


In [ ]:
from sklearn.metrics import r2_score
import numpy as np

y_pred = model.predict(X_test).flatten()

r2 = r2_score(y_test, y_pred)
mae = np.mean(np.abs(y_test - y_pred))
rmse = np.sqrt(np.mean((y_test - y_pred)**2))

# دقت به سبک classification (تلرانس ±5)
tolerance = 5
accuracy = np.mean(np.abs(y_test - y_pred) <= tolerance) * 100

print(f"R² Score: {r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"Accuracy (±5 tolerance): {accuracy:.2f}%")


In [ ]:
for tol in [1, 2, 3, 5, 10, 15]:
    acc = np.mean(np.abs(y_test - y_pred) <= tol) * 100
    print(f"Accuracy (±{tol}): {acc:.2f}%")


In [ ]:
import matplotlib.pyplot as plt

errors = y_test - y_pred
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.hist(errors, bins=50, color='steelblue', edgecolor='white')
plt.title('Error Distribution')
plt.xlabel('Error')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
plt.scatter(y_test, y_pred, alpha=0.1, color='steelblue')
plt.plot([0, 100], [0, 100], 'r--')
plt.title('Predicted vs Actual')
plt.xlabel('Actual')
plt.ylabel('Predicted')

plt.tight_layout()
plt.show()

In [ ]:
# Accuracy با tolerance های مختلف
for tol in [2, 5, 10]:
    acc = np.mean(np.abs(y_test - y_pred) <= tol) * 100
    print(f"Accuracy (±{tol}): {acc:.2f}%")

# Residual plot
import matplotlib.pyplot as plt
residuals = y_test - y_pred
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.scatter(y_pred, residuals, alpha=0.3)
plt.axhline(0, color='red')
plt.xlabel("Predicted")
plt.ylabel("Residual")
plt.title("Residual Plot")

plt.subplot(1, 2, 2)
plt.scatter(y_test, y_pred, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted")
plt.tight_layout()
plt.show()
